#  01) Find Gaia stars in DDF

- creation date : 2026-04-08
- author : Sylvie Dagoret-Campagne
- affiliation : IJCLab/IN2P3/CNRS

## 1. Imports & configuration

In [ ]:
import requests
import pandas as pd
import numpy as np
import json
import os
import time
import matplotlib.pyplot as plt
import warnings
from astroquery.gaia import Gaia

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
# Enable interactive matplotlib backend with zoom/pan toolbar
# Requires: pip install ipympl
# If ipympl is not available, fall back to inline (no interactivity)
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline (no zoom widget)")
    print("Install with:  pip install ipympl")

```
| **Name of Field**    | **RA(Degrees)** | **Dec (Degress)**| **Type**          |
| -------------------- | --------------- | ---------------- | ----------------- |
| **Carina**           | 161.5           | -59.7            | Galaxie/Nébuleuse |
| **ELAIS-S1**         | 9.45            | -44.0            | DDF               |
| **COSMOS**           | 150.1           | +2.2             | DDF               |
| **Trifid-Lagoon**    | 270.5           | -23.0            | Nébuleuse         |
| **M49**              | 187.4           | +8.0             | Galaxie           |
| **Rubin_SV_280_-48** | 280.0           | -48.0            | Test SV           |
| **Rubin_SV_320_-15** | 320.0           | -15.0            | Test SV           |
| **Rubin_SV_216_-17** | 216.0           | -17.0            | Test SV           |
| **Rubin_SV_212_-7**  | 212.0           | -7.0             | Test SV           |
| **Rubin_SV_225_-40** | 225.0           | -40.0            | Test SV           |
```

### DDF definition

In [ ]:
CONE_RADIUS = 1800.0  # Cone search radius in arcsec (0.5 deg per DDF)

BANDS = list("ugrizy")

# LSST Deep Drilling Fields (RA/Dec J2000)
DEEP_FIELDS = {
    "COSMOS": (150.1191, 2.2058),
    "ELAIS-S1": (9.4500, -44.000),
    "XMM-LSS": (35.7080, -4.750),
    "ECDFS": (53.1250, -27.8),
    "EDFS-a": (58.9, -49.315),
    "EDFS-b": (63.6, -47.6),
    "EDFS": (61.24, -48.423),
    "M49": (187.4, 8.0),
}

## 1. Gaia

### 1.1 Tables in Gaia

In [ ]:
query = """
SELECT table_name
FROM TAP_SCHEMA.tables
WHERE table_name LIKE 'gaiadr3.%'
"""

job = Gaia.launch_job_async(query)
tables = job.get_results().to_pandas()

print(tables.head(20))

### 1.2  Check Tables on variation in Gaia

In [ ]:
tables[tables["table_name"].str.contains("vari")]

#### 1.2.1 Check the table on Variation exists

In [ ]:
Gaia.launch_job_async("SELECT TOP 1 * FROM gaiadr3.vari_summary")

#### 1.2.2 Find the columns

In [ ]:
query = """
SELECT TOP 10 *
FROM gaiadr3.vari_summary
"""

job = Gaia.launch_job_async(query)
df = job.get_results().to_pandas()

In [ ]:
list(df.columns)

In [ ]:
query = """
SELECT TOP 1 *
FROM gaiadr3.vari_summary
"""
job = Gaia.launch_job(query)
result = job.get_results()
df = result.to_pandas()
print(df.columns)  # Affiche les noms des colonnes

### 1.3 Source Table

In [ ]:
from astroquery.gaia import Gaia

query = """
SELECT
    source_id,
    ra, dec,
    phot_g_mean_mag,
    bp_rp,
    parallax
FROM gaiadr3.gaia_source
WHERE
    CONTAINS(
        POINT('ICRS', ra, dec),
        CIRCLE('ICRS', 150.0, 2.0, 1.8)
    ) = 1
AND phot_g_mean_mag < 21
"""

job = Gaia.launch_job_async(query)
result = job.get_results()

df = result.to_pandas()

print(df.head())

In [ ]:
len(df)

### 1.4 Join of 2 tables `gaiadr3.gaia_source` and `gaiadr3.vari_summary`

#### 1.4.1 Simple join of 2 tables `gaiadr3.gaia_source` and `gaiadr3.vari_summary`

In [ ]:
query = """
SELECT
    g.source_id,
    g.ra, g.dec,
    g.phot_g_mean_mag,
    v.mean_mag_g_fov,
    v.median_mag_g_fov,
    v.std_dev_mag_g_fov,
    v.in_vari_long_period_variable,
    v.in_vari_eclipsing_binary,
    v.in_vari_classification_result
FROM gaiadr3.gaia_source AS g
LEFT JOIN gaiadr3.vari_summary AS v ON g.source_id = v.source_id
WHERE
    1 = CONTAINS(
        POINT('ICRS', g.ra, g.dec),
        CIRCLE('ICRS', 150.0, 2.0, 1.01)
    )
    AND g.phot_g_mean_mag < 20
    AND g.phot_g_mean_flux_over_error > 20
    AND g.ruwe < 1.4
"""
job = Gaia.launch_job(query)
df = job.get_results().to_pandas()

In [ ]:
df

#### 1.4.2 Complex join of 2 tables `gaiadr3.gaia_source` and `gaiadr3.vari_summary`

In [ ]:
query = """
WITH stable_candidates AS (
    SELECT
        g.source_id,
        g.ra, g.dec,
        g.phot_g_mean_mag,
        g.phot_g_mean_flux_over_error,
        g.ruwe
    FROM gaiadr3.gaia_source AS g
    WHERE
        1 = CONTAINS(
            POINT('ICRS', g.ra, g.dec),
            CIRCLE('ICRS', 150.0, 2.0, 1.01)
        )
        AND g.phot_g_mean_mag < 20
        AND g.phot_g_mean_flux_over_error > 20
        AND g.ruwe < 1.4
        -- Exclure les étoiles avec une entrée dans vari_summary (potentiellement variables)
        AND NOT EXISTS (
            SELECT 1
            FROM gaiadr3.vari_summary AS v
            WHERE v.source_id = g.source_id
        )
)
SELECT
    sc.*,
    -- Ajouter des métriques de stabilité si nécessaire (ex: depuis gaiadr3.epoch_photometry)
    ep.std_dev_mag_g AS photometric_stability
FROM stable_candidates AS sc
LEFT JOIN (
    -- Exemple : calculer l'écart-type de la magnitude G pour chaque source (si disponible)
    -- Remplace cette sous-requête par tes propres métriques si tu as accès à d'autres tables
    SELECT
        source_id,
        STDDEV(phot_g_mean_mag) AS std_dev_mag_g
    FROM gaiadr3.epoch_photometry
    GROUP BY source_id
) AS ep ON sc.source_id = ep.source_id
WHERE
    -- Filtre supplémentaire pour la stabilité photométrique (ex: écart-type < 0.05)
    (ep.std_dev_mag_g < 0.05 OR ep.source_id IS NULL)
    -- Exclure explicitement les binaires (si tu as une table dédiée, sinon utiliser vari_summary)
    AND NOT EXISTS (
        SELECT 1
        FROM gaiadr3.vari_summary AS vs
        WHERE vs.source_id = sc.source_id
        AND vs.in_vari_eclipsing_binary = 1
    )
"""
# job = Gaia.launch_job(query)
# df = job.get_results().to_pandas()

In [ ]:
query = """
SELECT
    g.source_id,
    g.ra, g.dec,
    g.phot_g_mean_mag
FROM gaiadr3.gaia_source AS g
WHERE
    1 = CONTAINS(
        POINT('ICRS', g.ra, g.dec),
        CIRCLE('ICRS', 150.0, 2.0, 1.01)
    )
    AND g.phot_g_mean_mag < 20
    AND g.phot_g_mean_flux_over_error > 20
    AND g.ruwe < 1.4
    -- Exclure les étoiles avec une entrée dans vari_summary
    AND NOT EXISTS (
        SELECT 1
        FROM gaiadr3.vari_summary AS v
        WHERE v.source_id = g.source_id
    )
    -- Exclure explicitement les binaires
    AND NOT EXISTS (
        SELECT 1
        FROM gaiadr3.vari_summary AS vs
        WHERE vs.source_id = g.source_id
        AND vs.in_vari_eclipsing_binary = 1
    )
"""

# job = Gaia.launch_job(query)
# df = job.get_results().to_pandas()

In [ ]:
query = """
SELECT
    g.source_id,
    g.ra, g.dec,
    g.phot_g_mean_mag,
    g.phot_g_mean_flux_over_error,
    g.ruwe
FROM gaiadr3.gaia_source AS g
WHERE
    1 = CONTAINS(
        POINT('ICRS', g.ra, g.dec),
        CIRCLE('ICRS', 150.0, 2.0, 1.01)
    )
    AND g.phot_g_mean_mag < 20
    AND g.phot_g_mean_flux_over_error > 20
    AND g.ruwe < 1.4
"""
job = Gaia.launch_job(query)
df = job.get_results().to_pandas()

# Ensuite, filtrer localement avec Python pour les étoiles stables et non binaires
# (si tu as déjà téléchargé la table vari_summary ou si tu peux la croiser localement)

In [ ]:
query = """
SELECT
    g.source_id,
    g.ra, g.dec,
    g.phot_g_mean_mag,
    g.astrometric_params_solved,
    g.astrometric_gof_al,
    g.phot_bp_rp_excess_factor,
    v.mean_mag_g_fov,
    v.median_mag_g_fov,
    v.std_dev_mag_g_fov,
    v.in_vari_long_period_variable,
    v.in_vari_eclipsing_binary,
    v.in_vari_classification_result,
    v.in_vari_rrlyrae,
    v.in_vari_cepheid,
    v.in_vari_planetary_transit,
    v.in_vari_short_timescale,
    v.in_vari_long_period_variable,
    v.in_vari_eclipsing_binary,
    v.in_vari_rotation_modulation,
    v.in_vari_ms_oscillator,
    v.in_vari_agn,
    v.in_vari_microlensing, 
    v.in_vari_compact_companion     
FROM gaiadr3.gaia_source AS g
LEFT JOIN gaiadr3.vari_summary AS v ON g.source_id = v.source_id
WHERE
    1 = CONTAINS(
        POINT('ICRS', g.ra, g.dec),
        CIRCLE('ICRS', 150.0, 2.0, 1.5)
    )
    AND g.phot_g_mean_mag < 21
    AND g.phot_g_mean_flux_over_error > 5
    AND g.ruwe < 1.4
"""
job = Gaia.launch_job(query)
df = job.get_results().to_pandas()

In [ ]:
df

**Comment identifier les sources ponctuelles dans Gaia DR3 ?**
Pour distinguer les sources ponctuelles (étoiles) des sources étendues (galaxies, artefacts, etc.), voici les colonnes et critères que tu peux utiliser dans gaiadr3.gaia_source :

- 1. `astrometric_params_solved`

Valeur = 31 : La source a une solution astrométrique complète (typique des étoiles).
Valeur < 31 : La source est probablement étendue ou de mauvaise qualité.

- 2. `astrometric_gof_al et astrometric_gof_ac`

Valeur faible (ex: < 3) : Bon ajustement astrométrique (source ponctuelle).
Valeur élevée (ex: > 10) : Mauvais ajustement (source étendue ou problématique).

- 3. `phot_bp_rp_excess_factor`

Valeur proche de 1 : Excès de flux BP/RP attendu pour une étoile.
Valeur élevée ou faible : Peut indiquer une source étendue ou un problème de photométrie.

- 4. `ruwe (Renormalised Unit Weight Error)`

Valeur < 1.4 : Bon ajustement astrométrique (déjà utilisé dans ta requête).

Résumé

Utilise astrometric_params_solved, astrometric_gof_al, et phot_bp_rp_excess_factor pour identifier les sources ponctuelles.
Ajuste les seuils selon tes besoins (ex: astrometric_gof_al < 5 pour être moins strict).


In [ ]:
# 1. Sélectionner les sources sans variabilité détectée
stable_mask = (
    (df["in_vari_classification_result"].fillna(0) == 0)
    & (df["in_vari_rrlyrae"].fillna(0) == 0)
    & (df["in_vari_cepheid"].fillna(0) == 0)
    & (df["in_vari_planetary_transit"].fillna(0) == 0)
    & (df["in_vari_short_timescale"].fillna(0) == 0)
    & (df["in_vari_long_period_variable"].fillna(0) == 0)
    & (df["in_vari_eclipsing_binary"].fillna(0) == 0)
    & (df["in_vari_rotation_modulation"].fillna(0) == 0)
    & (df["in_vari_ms_oscillator"].fillna(0) == 0)
    & (df["in_vari_agn"].fillna(0) == 0)
    & (df["in_vari_microlensing"].fillna(0) == 0)
    & (df["in_vari_compact_companion"].fillna(0) == 0)
)

# 2. Ajouter un critère de stabilité photométrique
photometric_stability_mask = df["std_dev_mag_g_fov"].fillna(0) < 0.05


# Critère 3.1 : Solution astrométrique complète (31 paramètres)
point_source_mask_params = df["astrometric_params_solved"] == 31

# Critère 3.2 : Bon ajustement astrométrique (astrometric_gof_al < 3)
point_source_mask_gof = df["astrometric_gof_al"] < 3

# Critère 3.3 : Excès de flux BP/RP raisonnable (entre 1 et 2)
point_source_mask_excess = (df["phot_bp_rp_excess_factor"] > 1) & (df["phot_bp_rp_excess_factor"] < 2)

# 3. Combiner les masques pour les sources ponctuelles
point_source_mask = point_source_mask_params & point_source_mask_gof & point_source_mask_excess


# 4. Combiner tous les masques
stable_sources_df = df[stable_mask & photometric_stability_mask & point_source_mask]

# 4. Combiner les masques
# stable_sources_df = df[stable_mask & photometric_stability_mask]

# 5. Afficher le nombre de sources stables sélectionnées
print(f"Nombre de sources ponctuelles stables : {len(stable_sources_df)}")
print(stable_sources_df.head())

In [ ]:
def plot_sources_ra_dec(
    df,
    magnitude_col="phot_g_mean_mag",
    size_scale=100,
    color_stable="blue",
    color_variable="red",
    title="Carte des sources (RA, Dec)",
    figsize=(12, 8),
    save_path=None,
):
    """
    Trace les sources en coordonnées (RA, Dec) avec une taille liée à la magnitude.

    Paramètres :
    -----------
    df : pandas.DataFrame
        DataFrame contenant les données des sources (doit inclure ra, dec, et la colonne de magnitude).
    magnitude_col : str, optionnel
        Nom de la colonne contenant la magnitude (par défaut 'phot_g_mean_mag').
    size_scale : float, optionnel
        Facteur d'échelle pour la taille des points (par défaut 100).
    color_stable : str, optionnel
        Couleur des sources stables (par défaut 'blue').
    color_variable : str, optionnel
        Couleur des sources variables (par défaut 'red').
    title : str, optionnel
        Titre du graphique (par défaut "Carte des sources (RA, Dec)").
    figsize : tuple, optionnel
        Taille de la figure (par défaut (12, 8)).
    save_path : str, optionnel
        Chemin pour sauvegarder le graphique (par défaut None, ne sauvegarde pas).
    """

    # Vérifier que les colonnes nécessaires sont présentes
    required_cols = ["ra", "dec", magnitude_col]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"La colonne '{col}' est manquante dans le DataFrame.")

    # Créer la figure
    plt.figure(figsize=figsize)

    # Inverser la magnitude pour que les étoiles brillantes soient plus grosses
    # (plus la magnitude est faible, plus l'étoile est brillante)
    sizes = size_scale * 10 ** (-0.4 * df[magnitude_col]) * 1e5

    # Tracer les sources stables (exemple : celles sans variabilité)
    # (Adapte cette condition selon tes critères de stabilité)
    stable_mask = (df["in_vari_classification_result"].fillna(0) == 0) & (
        df["in_vari_eclipsing_binary"].fillna(0) == 0
    )
    plt.scatter(
        df[~stable_mask]["ra"],
        df[~stable_mask]["dec"],
        s=sizes[~stable_mask],
        c=color_variable,
        alpha=0.6,
        label="Variables",
    )
    plt.scatter(
        df[stable_mask]["ra"],
        df[stable_mask]["dec"],
        s=sizes[stable_mask],
        c=color_stable,
        alpha=0.6,
        label="Stables",
    )

    # Ajouter des labels et un titre
    plt.xlabel("RA (degrees)")
    plt.ylabel("Dec (degrees)")
    plt.title(title)
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend()

    # Inverser l'axe des x pour que RA augmente de droite à gauche (convention astronomique)
    plt.gca().invert_xaxis()

    # Sauvegarder le graphique si un chemin est fourni
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    # Afficher le graphique
    plt.show()

In [ ]:
len(df)

In [ ]:
# Appeler la fonction avec ton DataFrame
plot_sources_ra_dec(
    df,
    magnitude_col="phot_g_mean_mag",
    size_scale=20,  # Ajuste ce paramètre pour changer la taille des points
    color_stable="blue",
    color_variable="red",
    title="Gaia map sources in COSMOS region (150°, 2°)",
    save_path="sources_cosmos_ra_dec.png",  # Optionnel : sauvegarde le graphique
)

In [ ]:
# Appeler la fonction avec ton DataFrame
plot_sources_ra_dec(
    stable_sources_df,
    magnitude_col="phot_g_mean_mag",
    size_scale=20,  # Ajuste ce paramètre pour changer la taille des points
    color_stable="blue",
    color_variable="red",
    title="Gaia map stable sources in COSMOS region (150°, 2°)",
    save_path="sources_cosmos_ra_dec.png",  # Optionnel : sauvegarde le graphique
)